# Лабораторная работа №5

В данной лабораторной работе мы изучим ансамбли моделей машинного обучения.

In [3]:
import numpy as np
import pandas as pd
from typing import Dict, Tuple
from scipy import stats
from IPython.display import Image
from io import StringIO 
from IPython.display import Image
import graphviz
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_graphviz
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score 
from sklearn.metrics import roc_curve, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline 
sns.set(style="ticks")

## Задание

* Выберите набор данных (датасет) для решения задачи классификации или регресии.
* В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.
* С использованием метода train_test_split разделите выборку на обучающую и тестовую.
* Обучите следующие ансамблевые модели:
  - две модели группы бэггинга (бэггинг или случайный лес или сверхслучайные деревья);
  - AdaBoost;
  - градиентный бустинг.
* Оцените качество моделей с помощью одной из подходящих для задачи метрик. Сравните качество полученных моделей.

## Датасет

Для лабораторной работы выберем датасет дрожжей: [https://archive.ics.uci.edu/dataset/110/yeast](ссылка). Перед нами задача классификации - распознать место локализация по ряду признаков образца.

Для удобства работы файл с данными отредактирован, чтобы первая строка содержала заголовки столбцов.

In [6]:
data = pd.read_csv('data/yeast/yeast.csv', sep=",")

# Отбросим признак Sequence_Name - идентификаторы предсказательной силы не имеют.
data = data.drop('Sequence_Name', axis=1)

In [7]:
# Форма датасета

data.shape

(1484, 9)

In [8]:
# Статистика по датасету

data.describe()

,mcg,gvh,alm,mit,erl,pox,vac,nuc
count,1484.000000,1484.000000,1484.000000,1484.000000,1484.000000,1484.000000,1484.000000,1484.000000
mean,0.500121,0.499933,0.500034,0.261186,0.504717,0.007500,0.499885,0.276199
std,0.137299,0.123924,0.086670,0.137098,0.048351,0.075683,0.057797,0.106491
min,0.110000,0.130000,0.210000,0.000000,0.500000,0.000000,0.000000,0.000000
25%,0.410000,0.420000,0.460000,0.170000,0.500000,0.000000,0.480000,0.220000
50%,0.490000,0.490000,0.510000,0.220000,0.500000,0.000000,0.510000,0.220000
75%,0.580000,0.570000,0.550000,0.320000,0.500000,0.000000,0.530000,0.300000
max,1.000000,1.000000,1.000000,1.000000,1.000000,0.830000,0.730000,1.000000


In [9]:
# Первые 5 строк

data.head()

,mcg,gvh,alm,mit,erl,pox,vac,nuc,localization_site
0,0.58,0.61,0.47,0.13,0.5,0.0,0.48,0.22,MIT
1,0.43,0.67,0.48,0.27,0.5,0.0,0.53,0.22,MIT
2,0.64,0.62,0.49,0.15,0.5,0.0,0.53,0.22,MIT
3,0.58,0.44,0.57,0.13,0.5,0.0,0.54,0.22,NUC
4,0.42,0.44,0.48,0.54,0.5,0.0,0.48,0.22,MIT


Пропусков данных нет. Выделим целевой признак в отдельный объект.

In [10]:
target = data['localization_site']
data = data.drop('localization_site', axis=1)

data.head()

,mcg,gvh,alm,mit,erl,pox,vac,nuc
0,0.58,0.61,0.47,0.13,0.5,0.0,0.48,0.22
1,0.43,0.67,0.48,0.27,0.5,0.0,0.53,0.22
2,0.64,0.62,0.49,0.15,0.5,0.0,0.53,0.22
3,0.58,0.44,0.57,0.13,0.5,0.0,0.54,0.22
4,0.42,0.44,0.48,0.54,0.5,0.0,0.48,0.22


## Разбиение датасета

Разобьем данные на две выборки: обучающую и тестовую. Пропорция: ~1/5 - тестовые данные, ~4/5 - обучающие.

In [11]:
# Разделение выборки на обучающую и тестовую
# Стратификация используется для сохранения распределения
yeast_X_train, yeast_X_test, yeast_y_train, yeast_y_test = train_test_split(
    data, target, test_size=0.2, random_state=1, stratify=target)

## Модели группы бэггинга

Обучим модель бэггинга.

In [12]:
bc1 = BaggingClassifier(n_estimators=100, oob_score=True, random_state=1)
bc1.fit(yeast_X_train, yeast_y_train)

,estimator,None
,n_estimators,100
,max_samples,1.0
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,True
,warm_start,False
,n_jobs,None
,random_state,1
,verbose,0


Обучим модель случайного леса.

In [13]:
bc2 = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=1)
bc2.fit(yeast_X_train, yeast_y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


## Модель группы бустинга

Обучим модель AdaBoost.

In [15]:
ab1 = AdaBoostClassifier(n_estimators=100, random_state=1)
ab1.fit(yeast_X_train, yeast_y_train)

,estimator,None
,n_estimators,100
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,1


Обучим модель градиентого бустинга. Используем реализацию scikit-learn, несмотря на её неэффективность. Это не помешает нам - датасет относительно небольшой.

In [16]:
gb1 = GradientBoostingClassifier(random_state=1)
gb1.fit(yeast_X_train, yeast_y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


## Оценка качества моделей

В нашем распоряжении имеются все требуемые модели. Оценим результат обучения с помощью F1-меры (macro) - так учтем дизбаланс классов.

In [23]:
# Список моделей
models = {
    'Bagging': bc1,
    'Random Forest': bc2,
    'AdaBoost': ab1,
    'Gradient Boosting': gb1
}

print("Сравнение моделей:")
print("-" * 50)
for name, model in models.items():
    pred = model.predict(yeast_X_test)
#    acc = accuracy_score(yeast_y_test, pred)
    f1 = f1_score(yeast_y_test, pred, average='macro')
    print(f"{name:20} | F1-score (macro): {f1:.4f}")

Сравнение моделей:
--------------------------------------------------
Bagging              | F1-score (macro): 0.4141
Random Forest        | F1-score (macro): 0.4146
AdaBoost             | F1-score (macro): 0.3126
Gradient Boosting    | F1-score (macro): 0.4507


### Вывод

Наилучшее качество показала модель градиентного бустинга, что объясняется последовательным характером обучения, позволяющим эффективно исправлять ошибки на сложных данных.

Модели бэггинга продемонстрировали практически идентичное качество. Это говорит о том, что для данного набора данных дополнительная рандомизация признаков в случайном лесе не дает значительного преимущества перед классическим бэггингом.

AdaBoost показал наихудший результат с большим отрывом. Вероятной причиной является чувствительность этого метода к шуму в данных, что характерно для задач с нечеткими биологическими признаками.